## Loading and Labelling the Data

In [1]:
import zipfile
import os

# List of files to unzip
zip_files = [
    'images_training_rev1.zip', 
    'images_test_rev1.zip', 
    'training_solutions_rev1.zip'
]

base_input_path = '/kaggle/input/competitions/galaxy-zoo-the-galaxy-challenge'
base_extract_path = '/kaggle/working/'

target_thresh = 0.8 #threshold for the data
RANDOM_SEED = 42

for file in zip_files:
    zip_path = os.path.join(base_input_path, file)
    
    # Create a specific folder for images, but keep the CSV in the main working dir
    if 'images' in file:
        folder_name = file.replace('.zip', '')
        extract_path = os.path.join(base_extract_path, folder_name)
    else:
        extract_path = base_extract_path
        
    os.makedirs(extract_path, exist_ok=True)

    print(f"Unzipping {file}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("All files extracted successfully.")

    location_path = os.path.join(base_extract_path, folder_name, folder_name)
    if os.path.exists(location_path):
        files = os.listdir(location_path)
        print(f"✅ Folder found!")
        print(f"Total files in training folder: {len(files)}")
        if len(files) > 0:
            print(f"Sample filename: {files[0]}")
    else:
        print("❌ Folder not found. Check your extraction path.")

Unzipping images_training_rev1.zip...
All files extracted successfully.
✅ Folder found!
Total files in training folder: 61578
Sample filename: 966634.jpg
Unzipping images_test_rev1.zip...
All files extracted successfully.
✅ Folder found!
Total files in training folder: 79975
Sample filename: 692093.jpg
Unzipping training_solutions_rev1.zip...
All files extracted successfully.
✅ Folder found!
Total files in training folder: 79975
Sample filename: 692093.jpg


In [2]:
#Hierarchical Thresholding

def thresh_labels(row, thresh):
    #the standards for the thresholds are 0.8 (from Willett et al. (2013), 0.5, and 0.7 (for the further tasks))
    # Scaled in this definition since lower quality images can be insightful testing data
    # Rule 1: High Consensus Smooth = Elliptical
    if row['Class1.1'] >= thresh:
        return 'Elliptical'
    
    # Rule 2: High Consensus Features + Arms = Spiral
    # (Note: We ignore the small % of 'Smooth' votes here)
    if row['Class1.2'] >= thresh and row['Class4.1'] >= ((5/8)*thresh):
        return 'Spiral'
    
    # Rule 3: High Consensus Artifact or 'Odd' = Irregular
    if row['Class1.3'] >= thresh or row['Class6.1'] >= ((7/8)*thresh):
        return 'Irregular'
    
    # Rule 4: If no consensus, throw it out! (Prevents noisy training)
    return 'Uncertain'

In [3]:
import pandas as pd

training_sol = pd.read_csv('/kaggle/working/training_solutions_rev1.csv')

# 3. Apply the function to create a new column
# 'args' passes the thresh value to your function
training_sol['label'] = training_sol.apply(thresh_labels, axis=1, thresh=target_thresh)

# 4. Filter out the 'Uncertain' rows to get your final labeled DF
train_sol_labeled = training_sol[training_sol['label'] != 'Uncertain'].copy()

# 5. Check the results
print(f"Original size: {len(training_sol)}")
print(f"Labeled size: {len(train_sol_labeled)}")
print(train_sol_labeled['label'].value_counts())

Original size: 61578
Labeled size: 20529
label
Spiral        9107
Elliptical    8132
Irregular     3290
Name: count, dtype: int64


## The Stratified 80/10/10 Split

In [4]:
from sklearn.model_selection import train_test_split

# Create the 80/10/10 split
# 1. First split: 80% Train, 20% "Remainder"
train_df, temp_df = train_test_split(
    train_sol_labeled, 
    test_size=0.2, 
    stratify=train_sol_labeled['label'], 
    random_state=RANDOM_SEED
)

# 2. Second split: Split the 20% "Remainder" into half (10% Val, 10% Test)
# FIX: Use temp_df here, not the full train_sol_labeled
val_df, test_df = train_test_split(
    temp_df, 
    test_size=0.5, 
    stratify=temp_df['label'], 
    random_state=RANDOM_SEED
)

## Image Integrity

In [5]:
import cv2
import os
from tqdm import tqdm # Great for tracking progress on 20k images

def check_image_integrity(df, img_path):
    corrupted_ids = []
    missing_ids = []
    
    print(f"Checking integrity for {len(df)} images...")
    
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        galaxy_id = int(row['GalaxyID'])
        full_path = os.path.join(img_path, f"{galaxy_id}.jpg")
        
        # 1. Check if file exists
        if not os.path.exists(full_path):
            missing_ids.append(galaxy_id)
            continue
            
        # 2. Try to decode the image
        img = cv2.imread(full_path)
        
        # 3. Check if image is corrupted or empty
        if img is None or img.size == 0:
            corrupted_ids.append(galaxy_id)
            
    return missing_ids, corrupted_ids

# Run the check
missing, corrupted = check_image_integrity(train_sol_labeled, '/kaggle/working/images_training_rev1/images_training_rev1')

print(f"\nResults:")
print(f"Missing Files: {len(missing)}")
print(f"Corrupted Files: {len(corrupted)}")

Checking integrity for 20529 images...


100%|██████████| 20529/20529 [00:21<00:00, 951.95it/s]


Results:
Missing Files: 0
Corrupted Files: 0


In [6]:
bad_ids = missing + corrupted

if bad_ids:
    print(f"Removing {len(bad_ids)} problematic images from splits...")
    train_df = train_df[~train_df['GalaxyID'].isin(bad_ids)]
    val_df = val_df[~val_df['GalaxyID'].isin(bad_ids)]
    test_df = test_df[~test_df['GalaxyID'].isin(bad_ids)]
else:
    print("All images verified! Dataset is healthy.")

All images verified! Dataset is healthy.


## Calculate "Space-Specific" Normalization

In [7]:
import numpy as np
import cv2
import os
from tqdm import tqdm

def calculate_galaxy_stats(df, img_path):
    # Initialize sums for Mean and Std
    # We use 3 channels (R, G, B)
    pixel_sum = np.zeros(3)
    pixel_sq_sum = np.zeros(3)
    pixel_count = 0
    
    print("Calculating Space-Specific Normalization Stats...")
    
    for _, row in tqdm(df.iterrows(), total=len(df)):
        img_id = int(row['GalaxyID'])
        img = cv2.imread(os.path.join(img_path, f"{img_id}.jpg"))
        
        if img is not None:
            # Convert BGR (OpenCV) to RGB and normalize to [0, 1]
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) / 255.0
            
            # Add to global sums
            pixel_sum += np.mean(img, axis=(0, 1))
            pixel_sq_sum += np.mean(np.square(img), axis=(0, 1))
            pixel_count += 1
            
    # Final Mean and Std calculation
    mean = pixel_sum / pixel_count
    std = np.sqrt((pixel_sq_sum / pixel_count) - np.square(mean))
    
    return mean, std

# Run on Training Set Only
actual_img_path = '/kaggle/working/images_training_rev1/images_training_rev1'
train_mean, train_std = calculate_galaxy_stats(train_df, actual_img_path)

print(f"\nSpace-Specific Mean: {train_mean}")
print(f"Space-Specific Std:  {train_std}")

Calculating Space-Specific Normalization Stats...


100%|██████████| 16423/16423 [02:56<00:00, 92.87it/s]


Space-Specific Mean: [0.04626448 0.04110195 0.03094786]
Space-Specific Std:  [0.08966795 0.07532379 0.06744124]


## Saving the CSVs

In [8]:
# 1. Save the Split CSVs
train_df.to_csv('train_split.csv', index=False)
val_df.to_csv('val_split.csv', index=False)
test_df.to_csv('test_split.csv', index=False)

# 2. Save your Space-Specific Normalization Stats (for easy copy-pasting)
stats = {
    "mean": train_mean.tolist(),
    "std": train_std.tolist()
}

import json
with open('norm_stats.json', 'w') as f:
    json.dump(stats, f)

print("Successfully exported Master Index and Normalization Stats!")
print(f"Final Training Samples: {len(train_df)}")
print(f"Final Validation Samples: {len(val_df)}")
print(f"Final Testing Samples: {len(test_df)}")

Successfully exported Master Index and Normalization Stats!
Final Training Samples: 16423
Final Validation Samples: 2053
Final Testing Samples: 2053
